### About the dataset

This dataset shows Rossmann's store sales, which are influenced by many factors, like promotions, competition, school and state holidays, seasonality and location. 

In this project I have to predict 6 weeks of daily sales for 1,115 stores located across Germany. And by buiding reliable forecasting model I will enable Rossmann's managers to organize staff schedules, delivery and promotion planning more effectively, which will increase productivity and sales.

Without model finding patterns and make decisions for efficient store management will be extremely hard because of the amount of factors which human can't consider.

---

Dataset by itself divided into 4 parts. In this notebook I'll work with train and store parts which contains train data and other useful features for model training

### Import & First look analysis

In [1]:
import pandas as pd

In [2]:
train = pd.read_csv("../data/raw/train.csv")
store = pd.read_csv("../data/raw/store.csv")

C:\Users\babuk\AppData\Local\Temp\ipykernel_37868\2043513621.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv("../data/raw/train.csv")


In [3]:
df = pd.merge(train, store, how="left", on="Store")

In [4]:
df.shape

(1017209, 18)

In [5]:
df.describe(include='all')

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
count,1.017209e+06,1.017209e+06,1017209,1.017209e+06,1.017209e+06,1.017209e+06,1.017209e+06,1017209,1.017209e+06,1017209,1017209,1.014567e+06,693861.000000,693861.000000,1.017209e+06,509178.000000,509178.000000,509178
unique,NaN,NaN,942,NaN,NaN,NaN,NaN,5,NaN,4,3,NaN,NaN,NaN,NaN,NaN,NaN,3
top,NaN,NaN,2013-01-02,NaN,NaN,NaN,NaN,0,NaN,a,a,NaN,NaN,NaN,NaN,NaN,NaN,"Jan,Apr,Jul,Oct"
freq,NaN,NaN,1115,NaN,NaN,NaN,NaN,855087,NaN,551627,537445,NaN,NaN,NaN,NaN,NaN,NaN,293122
mean,5.584297e+02,3.998341e+00,NaN,5.773819e+03,6.331459e+02,8.301067e-01,3.815145e-01,NaN,1.786467e-01,NaN,NaN,5.430086e+03,7.222866,2008.690228,5.005638e-01,23.269093,2011.752774,NaN
std,3.219087e+02,1.997391e+00,NaN,3.849926e+03,4.644117e+02,3.755392e-01,4.857586e-01,NaN,3.830564e-01,NaN,NaN,7.715324e+03,3.211832,5.992644,4.999999e-01,14.095973,1.662870,NaN
min,1.000000e+00,1.000000e+00,NaN,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,NaN,0.000000e+00,NaN,NaN,2.000000e+01,1.000000,1900.000000,0.000000e+00,1.000000,2009.000000,NaN
25%,2.800000e+02,2.000000e+00,NaN,3.727000e+03,4.050000e+02,1.000000e+00,0.000000e+00,NaN,0.000000e+00,NaN,NaN,7.100000e+02,4.000000,2006.000000,0.000000e+00,13.000000,2011.000000,NaN
50%,5.580000e+02,4.000000e+00,NaN,5.744000e+03,6.090000e+02,1.000000e+00,0.000000e+00,NaN,0.000000e+00,NaN,NaN,2.330000e+03,8.000000,2010.000000,1.000000e+00,22.000000,2012.000000,NaN
75%,8.380000e+02,6.000000e+00,NaN,7.856000e+03,8.370000e+02,1.000000e+00,1.000000e+00,NaN,0.000000e+00,NaN,NaN,6.890000e+03,10.000000,2013.000000,1.000000e+00,37.000000,2013.000000,NaN


In [6]:
df.isna().sum()

Store                             0
DayOfWeek                         0
Date                              0
Sales                             0
Customers                         0
Open                              0
Promo                             0
StateHoliday                      0
SchoolHoliday                     0
StoreType                         0
Assortment                        0
CompetitionDistance            2642
CompetitionOpenSinceMonth    323348
CompetitionOpenSinceYear     323348
Promo2                            0
Promo2SinceWeek              508031
Promo2SinceYear              508031
PromoInterval                508031
dtype: int64

We see that some of the data, that represents competition distance, open month and open year are missing
- this is because those features represent competition stores information that located nearby. Some of Rossmann's stores just don't have any competetors, so they have NaN values

Also some Promo2 values missing
- this is because some of the stores didn't participate in second promo so there's no informaton about it's beginnning time and interval, etc.

---

In [7]:
df.duplicated().any()

np.False_

There's no duplicates in Dataset

---

### Handling missing values

In [8]:
nan_cols = df.columns[df.isna().any()]
nan_cols

Index(['CompetitionDistance', 'CompetitionOpenSinceMonth',
       'CompetitionOpenSinceYear', 'Promo2SinceWeek', 'Promo2SinceYear',
       'PromoInterval'],
      dtype='object')

In [9]:
df[nan_cols].head(5)

,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1270.0,9.0,2008.0,NaN,NaN,NaN
1,570.0,11.0,2007.0,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,14130.0,12.0,2006.0,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,620.0,9.0,2009.0,NaN,NaN,NaN
4,29910.0,4.0,2015.0,NaN,NaN,NaN


---

With promo features I'll use fillna 0, because it means that the store never had second promo (but i'll fill Interval feature with `None` since it'll be encoded in the future)

In [10]:
df['Promo2SinceWeek'] = df['Promo2SinceWeek'].fillna(0)
df['Promo2SinceYear'] = df['Promo2SinceYear'].fillna(0)
df['PromoInterval'] = df['PromoInterval'].fillna('None')

---

In [11]:
df['CompetitionDistance'].describe()

count    1.014567e+06
mean     5.430086e+03
std      7.715324e+03
min      2.000000e+01
25%      7.100000e+02
50%      2.330000e+03
75%      6.890000e+03
max      7.586000e+04
Name: CompetitionDistance, dtype: float64

We see that distance varies from 20 metres all across to 75 kilometers, which means there's a lot of outliers

So I'll use median-type NA filling for outliers robustness and I'll also add missingness indicator as a feature.

In [12]:
df['CompetitionDistance_missing'] = (
    df['CompetitionDistance'].isna().astype(int)
)

In [13]:
df['CompetitionDistance'] = (
    df['CompetitionDistance'].fillna(
        df['CompetitionDistance'].median()
    )
)

---

In [14]:
df.head(5)

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval,CompetitionDistance_missing
0,1,5,2015-07-31,5263,555,1,1,0,1,c,a,1270.0,9.0,2008.0,0,0.0,0.0,None,0
1,2,5,2015-07-31,6064,625,1,1,0,1,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct",0
2,3,5,2015-07-31,8314,821,1,1,0,1,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct",0
3,4,5,2015-07-31,13995,1498,1,1,0,1,c,c,620.0,9.0,2009.0,0,0.0,0.0,None,0
4,5,5,2015-07-31,4822,559,1,1,0,1,a,a,29910.0,4.0,2015.0,0,0.0,0.0,None,0


For Competition opening date columns I'll create seperate feature that will show how much time competetor exists and add another feature which will be indicator of competition

Then I'll drop the original features.

In [15]:
competition_date = pd.to_datetime(
    df["CompetitionOpenSinceYear"].astype("Int64").astype(str)
    + "-"
    + df["CompetitionOpenSinceMonth"].astype("Int64").astype(str)
    + "-01",
    errors="coerce",
)

df["Date"] = pd.to_datetime(df["Date"]) #appeared not to be in dt format

In [16]:
df['competition_age_months'] = (
    (df['Date'].dt.year - competition_date.dt.year) * 12 +
    (df['Date'].dt.month - competition_date.dt.month)
)

In [17]:
df["competition_age_months"] = (
    df["competition_age_months"].fillna(0)
)

In [18]:
df["has_competition"] = competition_date.notna().astype(int)

In [19]:
df = df.drop(columns = ['CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear'])

In [20]:
df.isna().any().sum()

np.int64(0)

---

### Feature engineering

In [21]:
df["year"] = df["Date"].dt.year
df["month"] = df["Date"].dt.month
df["day"] = df["Date"].dt.day
df["week_of_year"] = df["Date"].dt.isocalendar().week.astype(int)

df['is_Saturday'] = (df['DayOfWeek'] == 6).astype(int)
df['is_Sunday'] = (df['DayOfWeek'] == 7).astype(int)

In [22]:
df = df.sort_values(["Store", "Date"])

df["sales_lag_1"] = (
    df.groupby("Store")["Sales"].shift(1)
)

df["sales_lag_7"] = (
    df.groupby("Store")["Sales"].shift(7)
)

df["sales_rolling_7"] = (
    df.groupby("Store")["Sales"]
      .transform(lambda x: x.shift(1).rolling(7).mean())
)

In [23]:
df = df.dropna(
    subset=[
        "sales_lag_1",
        "sales_lag_7",
        "sales_rolling_7"
    ]
)

In [24]:
df = df.drop(columns=['Date'])

I'm keeping Store columns because it's necessary for forecasting even though it has 1115 unique values.

But those values help to determine which store am i predicting.

---

### Encoding

In [25]:
df.select_dtypes(include="object").columns

Index(['StateHoliday', 'StoreType', 'Assortment', 'PromoInterval'], dtype='object')

In [26]:
cat_cols = ['StateHoliday', 'StoreType', 'Assortment', 'PromoInterval']

In [27]:
for col in df.select_dtypes(include="object").columns:
    print(f"{col}: {df[col].nunique()} unique values")

StateHoliday: 5 unique values
StoreType: 4 unique values
Assortment: 3 unique values
PromoInterval: 4 unique values


In [28]:
for col in df.select_dtypes(include="object").columns:
    print(f"\n{col}")
    print(df[col].value_counts(dropna=False))


StateHoliday
StateHoliday
0    848705
0    131072
a     18837
b      6690
c      4100
Name: count, dtype: int64

StoreType
StoreType
a    547413
d    310476
c    135804
b     15711
Name: count, dtype: int64

Assortment
Assortment
a    533294
c    467879
b      8231
Name: count, dtype: int64

PromoInterval
PromoInterval
None                504223
Jan,Apr,Jul,Oct     290777
Feb,May,Aug,Nov     117686
Mar,Jun,Sept,Dec     96718
Name: count, dtype: int64


In [29]:
df["StateHoliday"] = df["StateHoliday"].astype(str)

Here I noticed that StateHoliday has 0 as a int and 0 as a string so i converted it

In [30]:
df = pd.get_dummies(
    df,
    columns=cat_cols,
    dtype=int
)

In [31]:
df.columns[df.dtypes == "Object"]

Index([], dtype='object')

All of the features are finally numerical, there's no NaN and duplicates. 

Additional columns are created and all the useful information found and used.

Dataset is ready to be used in model, but first let's do EDA

In [ ]:
# df.to_csv('../data/processed/rossmannV2.csv', index=False)

Also i created two versions of datasets in which first one with `is_wekeend` feature 

And the second is without it but with two separate features instead (`is_saturday and is_sunday`)